# 01 – Data Collection & Initial Structural Verification (Milestone P2, nhóm N05)

Notebook này tái lập toàn bộ bước thu thập dữ liệu của Milestone 2:
1. Tải OULAD (nếu chưa có) từ nguồn đã ghi trong Data Acquisition Plan.
2. Kiểm tra toàn vẹn bằng SHA-256 so với `data/metadata/SHA256SUMS.txt`.
3. Kiểm tra cấu trúc tối thiểu (số dòng/cột, khóa, kiểu dữ liệu, ký hiệu thiếu, trường thời gian, đối chiếu tài liệu).

**Không** làm sạch, impute, chuẩn hóa hay feature engineering — các bước đó thuộc Milestone 3.

In [1]:
import hashlib, json, subprocess, sys, urllib.request, zipfile
from pathlib import Path
import pandas as pd
P2 = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW, META = P2/'data'/'raw', P2/'data'/'metadata'
URL_OFFICIAL = 'https://schools.stem.open.ac.uk/cdn/files/anonymisedData.zip'   # link trên trang OU Analyse (HTTP 404 ngày 09/11/2026)
URL_USED = 'https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip'  # UCI ML Repository, id 349
ZIP = RAW/'open+university+learning+analytics+dataset.zip'
print('Thư mục dự án:', P2.name)

Thư mục dự án: P2


## 1. Tải dữ liệu (bỏ qua nếu file đã có — raw không bao giờ bị ghi đè)

In [2]:
if not ZIP.exists():                                   # tải zip gốc nếu chưa có
    urllib.request.urlretrieve(URL_USED, ZIP)
if not (RAW/'oulad'/'studentVle.csv').exists():       # giải nén nếu chưa có (ví dụ sau khi clone repo)
    with zipfile.ZipFile(ZIP) as z: z.extractall(RAW/'oulad')
print(ZIP.name, ZIP.stat().st_size, 'bytes')
with zipfile.ZipFile(ZIP) as z:
    display(pd.DataFrame([(i.filename, i.file_size, '%04d-%02d-%02d' % i.date_time[:3]) for i in z.infolist()], columns=['file','bytes','date_in_zip']))

open+university+learning+analytics+dataset.zip 46748244 bytes


,file,bytes,date_in_zip
0,assessments.csv,8211,2015-12-21
1,courses.csv,526,2015-09-25
2,studentAssessment.csv,5690483,2015-12-21
3,studentInfo.csv,3462763,2015-12-21
4,studentRegistration.csv,1132550,2015-12-21
5,studentVle.csv,453836331,2015-09-25
6,vle.csv,270612,2015-12-21
7,OULAD.names,12278,2015-12-21


## 2. Kiểm tra toàn vẹn (SHA-256)

In [3]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
rows = []
for line in (META/'SHA256SUMS.txt').read_text().splitlines():
    exp, rel = line.split(maxsplit=1)
    got = sha256(RAW/rel)
    rows.append((rel, exp[:16]+'…', got == exp))
chk = pd.DataFrame(rows, columns=['file','sha256 (expected)','match'])
display(chk); assert chk['match'].all(), 'Checksum mismatch!'

,file,sha256 (expected),match
0,open+university+learning+analytics+dataset.zip,f2ed1902616c1fe8…,True
1,oulad/assessments.csv,9ce92381e4a0ac45…,True
2,oulad/courses.csv,4f16eee7454b15e1…,True
3,oulad/OULAD.names,e497bc5232f385ec…,True
4,oulad/studentAssessment.csv,b1839e17618a5de3…,True
5,oulad/studentInfo.csv,815fbc3e2de29f79…,True
6,oulad/studentRegistration.csv,bbc87a0de1fe2a9e…,True
7,oulad/studentVle.csv,52668253d876c5be…,True
8,oulad/vle.csv,d7a146497edd0e47…,True


## 3. Initial Structural Verification

In [4]:
subprocess.run([sys.executable, str(P2/'src'/'verify_structure.py')], check=True)
r = json.loads((META/'structural_verification.json').read_text())
display(pd.DataFrame([{'table': k, 'rows': t['rows'], 'rows_in_doc': t['doc_rows'], 'cols': t['cols'],
                       'key': ', '.join(t.get('key') or ['(log, no key)']), 'dup_key': t.get('dup_key_rows'),
                       'dup_full_rows': t['dup_full_rows']} for k, t in r['tables'].items()]))

courses (22, 3)
assessments (206, 6)
vle (6364, 6)
studentInfo (32593, 12)
studentRegistration (32593, 5)


studentAssessment (173912, 5)


studentVle (10655280, 6)


written data/metadata/structural_verification.json


,table,rows,rows_in_doc,cols,key,dup_key,dup_full_rows
0,courses,22,22,3,"code_module, code_presentation",0.0,0
1,assessments,206,206,6,id_assessment,0.0,0
2,vle,6364,6364,6,id_site,0.0,0
3,studentInfo,32593,32953,12,"code_module, code_presentation, id_student",0.0,0
4,studentRegistration,32593,32953,5,"code_module, code_presentation, id_student",0.0,0
5,studentAssessment,173912,173912,5,"id_assessment, id_student",0.0,0
6,studentVle,10655280,10655280,6,"(log, no key)",NaN,787170


In [5]:
# Ký hiệu thiếu trong file gốc ('?') theo cột
display(pd.DataFrame([{'table': k, 'column': c, 'n_qmark': i['n_qmark'], 'n_empty': i['n_empty']}
                      for k, t in r['tables'].items() for c, i in t['columns'].items() if i['n_qmark'] or i['n_empty']]))

,table,column,n_qmark,n_empty
0,assessments,date,11,0
1,vle,week_from,5243,0
2,vle,week_to,5243,0
3,studentInfo,imd_band,1111,0
4,studentRegistration,date_registration,45,0
5,studentRegistration,date_unregistration,22521,0
6,studentAssessment,score,173,0


In [6]:
for k in ['final_result_dist', 'integrity', 'doc_check_unreg', 'time_ranges']:
    print(k, json.dumps(r[k], ensure_ascii=False))
print('imd_band values:', r['value_sets']['imd_band'])

final_result_dist {"Pass": 12361, "Withdrawn": 10156, "Fail": 7052, "Distinction": 3024}
integrity {"studentInfo_vs_registration_same_keys": 32593, "studentVle_keys_not_in_studentInfo": 0, "studentVle_site_not_in_vle": 0, "studentAssessment_assess_not_in_assessments": 0, "students_in_info_without_any_vle": 3365, "unique_id_student": 28785}
doc_check_unreg {"has_unreg_date": 10072, "has_unreg_but_not_withdrawn": 9, "withdrawn_without_unreg_date": 93, "unreg_before_day0": 3097, "unreg_day1_28": 1958}
time_ranges {"studentVle.date": [-25, 269], "studentAssessment.date_submitted": [-11, 608], "assessments.date": [12.0, 261.0], "studentRegistration.date_registration": [-322.0, 167.0], "courses.length": [234, 269]}
imd_band values: ['0-10%', '10-20', '20-30%', '30-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']


## 4. Tổng quan theo module (căn cứ đề xuất phạm vi BBB + FFF)

In [7]:
cols = ['module','n_presentations','n_enrolments','nonCompletion_rate','first_nonexam_assessment_day_by_pres','share_with_vle_0_28','median_clicks_0_28','imd_missing_share']
display(pd.DataFrame(r['modules'])[cols])
display(pd.DataFrame(r['proposed_scope'])); print(r['proposed_scope_total'])

,module,n_presentations,n_enrolments,nonCompletion_rate,first_nonexam_assessment_day_by_pres,share_with_vle_0_28,median_clicks_0_28,imd_missing_share
0,AAA,2,748,0.2901,"{'2013J': 19.0, '2014J': 19.0}",0.9679,275.0,0.0307
1,BBB,4,7909,0.5254,"{'2013B': 19.0, '2013J': 19.0, '2014B': 12.0, ...",0.8017,100.0,0.0080
2,CCC,2,4434,0.6216,"{'2014B': 18.0, '2014J': 18.0}",0.8602,199.0,0.0573
3,DDD,4,6272,0.5839,"{'2013B': 23.0, '2013J': 25.0, '2014B': 25.0, ...",0.8763,162.0,0.0446
4,EEE,3,2934,0.4376,"{'2013J': 33.0, '2014B': 33.0, '2014J': 33.0}",0.8814,283.5,0.0399
5,FFF,4,7762,0.5300,"{'2013B': 19.0, '2013J': 19.0, '2014B': 24.0, ...",0.8817,390.5,0.0473
6,GGG,3,2534,0.4025,"{'2013J': 61.0, '2014B': 61.0, '2014J': 61.0}",0.8299,43.0,0.0028


,module,presentation,n_enrolments,nonCompletion_all,active_at_day28,nonCompletion_active_d28
0,BBB,2013B,1767,0.5456,1530,0.4752
1,BBB,2013J,2237,0.5208,1835,0.4158
2,BBB,2014B,1613,0.5493,1282,0.4329
3,BBB,2014J,2292,0.4974,1787,0.3553
4,FFF,2013B,1614,0.5155,1450,0.4607
5,FFF,2013J,2283,0.5204,1982,0.4475
6,FFF,2014B,1500,0.5640,1302,0.4977
7,FFF,2014J,2365,0.5277,1859,0.3991


{'n_enrolments': 15671, 'active_at_day28': 13027, 'nonCompletion_all': 0.5277, 'nonCompletion_active_d28': 0.4318}
